## Import and preprocessing

Import

In [79]:
%load_ext autoreload
%autoreload 2

from importlib import reload
import my_sklearn
reload(my_sklearn)

from my_sklearn import my_LinearRegression

import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Read train and test json files, downloaded from [kaggle.com](https://www.kaggle.com/competitions/two-sigma-connect-rental-listing-inquiries/data?select=train.json.zip)

In [3]:
df = pd.read_json("data/train.json").reset_index()

df.head()

,index,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
0,4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
1,6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
2,9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
3,10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
4,15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low


Generating additional features for better model quality

In [4]:
df['Features'] = df['features'].apply(
    lambda x: str(x)
    .replace('[', '')
    .replace(']', '')
    .replace('"', '')
    .replace("'", '')
    .replace(' ', '')
)

In [5]:
features = (
    df['Features']
    .str.split(',')
    .explode()
    .loc[lambda s: s != '']
    .tolist()
)
unique = set(features)
len(unique)

1546

In [6]:
top_feat = Counter(features).most_common(20)
top_feat

[('Elevator', 25915),
 ('CatsAllowed', 23540),
 ('HardwoodFloors', 23527),
 ('DogsAllowed', 22035),
 ('Doorman', 20898),
 ('Dishwasher', 20426),
 ('NoFee', 18062),
 ('LaundryinBuilding', 16344),
 ('FitnessCenter', 13252),
 ('Pre-War', 9148),
 ('LaundryinUnit', 8738),
 ('RoofDeck', 6542),
 ('OutdoorSpace', 5268),
 ('DiningRoom', 5136),
 ('HighSpeedInternet', 4299),
 ('Balcony', 2992),
 ('SwimmingPool', 2730),
 ('LaundryInBuilding', 2593),
 ('NewConstruction', 2559),
 ('Terrace', 2283)]

In [7]:
feature_list = [ftr[0] for ftr in top_feat]

X = pd.DataFrame()

for feature in feature_list:
    X[feature] = df['Features'].str.contains(feature, regex=False).astype('int8')

X['bathrooms'] = df['bathrooms']
X['bedrooms'] = df['bedrooms']

feature_list += ['bathrooms', 'bedrooms']

y = df['price']

## Models implementation

### Linear Regression

In [81]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=21,
    test_size=0.2
)

result_R2 = pd.DataFrame(columns=['model', 'train', 'test'])
result_MAE = pd.DataFrame(columns=['model', 'train', 'test'])
result_RMSE = pd.DataFrame(columns=['model', 'train', 'test'])

In [ ]:
def my_R2(y, y_pred):

    y = np.asarray(y)
    y_pred = np.asarray(y_pred)

    if y.shape != y_pred.shape:
        raise ValueError('must be the same shape')

    y_mean = np.mean(y)
    tss = np.sum((y - y_mean)**2)
    
    if np.isclose(tss, 0):
        raise ValueError("TSS must be not 0")
    
    rss = np.sum((y - y_pred)**2)

    return 1 - rss / tss

def evaluate_model(model, name, result_MAE, result_RMSE, result_R2, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    MAE_test = mean_absolute_error(y_test, y_pred_test)
    MAE_train = mean_absolute_error(y_train, y_pred_train)

    RMSE_test = root_mean_squared_error(y_test, y_pred_test)
    RMSE_train = root_mean_squared_error(y_train, y_pred_train)

    R2_train = my_R2(y_train, y_pred_train)
    R2_test = my_R2(y_test, y_pred_test)

    result_MAE.loc[len(result_MAE)] = [name, MAE_train, MAE_test]
    result_RMSE.loc[len(result_RMSE)] = [name, RMSE_train, RMSE_test]
    result_R2.loc[len(result_R2)] = [name, R2_train, R2_test]




#### sklearn linear regression

In [82]:
model = LinearRegression()
name = 'LinearRegression'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

#### my implementation

In [85]:
model = my_LinearRegression()
name = 'my_LinearRegression'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)